# Atelier — Construire un agent ReAct multi-outils avec Claude

**Objectif pédagogique** : comprendre et reproduire le pattern ReAct (Reasoning + Acting),
en partant du papier de recherche original, puis l'adapter à un nouveau cas d'usage
avec plusieurs outils (recherche + calcul) et le tool use natif de l'API Claude.

**Ce que vous allez apprendre** :
1. Ce qu'est la boucle Thought → Action → Observation
2. Pourquoi le parsing de texte brut est fragile, et comment le tool use structuré le résout
3. Comment brancher plusieurs outils différents sur un même agent
4. Comment tester chaque brique indépendamment avant de les assembler

**Prérequis** : une clé API Claude (`ANTHROPIC_API_KEY`), Python 3.9+, une connexion internet.


## Bloc 1 — Setup

On installe et importe ce qu'il faut, puis on initialise le client Claude.
**Test attendu** : cette cellule doit s'exécuter sans erreur et n'affiche rien —
c'est normal, elle ne fait qu'importer et configurer.


In [ ]:
import os
import ast
import operator
import requests
import anthropic

client = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
print("Client Claude initialisé.")


## Bloc 2 — Les outils de l'agent

Contrairement au notebook original de ReAct (qui n'utilisait qu'un seul outil,
`Search` sur Wikipédia), on en branche **deux**, pour montrer comment un agent
orchestre plusieurs capacités différentes :

- `search_wikipedia` : une vraie recherche sur Wikipédia (API publique, aucune clé requise)
- `calculer` : un calcul mathématique, exécuté avec un évaluateur **sécurisé**
  (jamais de `eval()` brut sur du texte généré par un LLM — c'est un point de
  sécurité important à retenir : un LLM peut halluciner ou être manipulé pour
  injecter du code)

**Test attendu** : chaque fonction sera testée isolément dans le bloc suivant,
avant même d'être branchée à l'agent.


In [ ]:
def search_wikipedia(query: str) -> str:
    """Cherche un terme exact sur Wikipédia (FR) et renvoie le résumé."""
    url = f"https://fr.wikipedia.org/api/rest_v1/page/summary/{query.replace(' ', '_')}"
    try:
        response = requests.get(url, timeout=5)
        if response.status_code == 200:
            return response.json().get("extract", "Aucun résumé trouvé.")
        return f"Aucune page trouvée pour '{query}'."
    except requests.exceptions.RequestException as e:
        return f"Erreur réseau : {e}"


_ALLOWED_OPERATORS = {
    ast.Add: operator.add,
    ast.Sub: operator.sub,
    ast.Mult: operator.mul,
    ast.Div: operator.truediv,
    ast.Pow: operator.pow,
    ast.USub: operator.neg,
}

def safe_calculate(expression: str) -> float:
    """Évalue une expression arithmétique simple, sans exécuter de code arbitraire."""
    def _eval(node):
        if isinstance(node, ast.Constant):
            return node.value
        if isinstance(node, ast.BinOp):
            return _ALLOWED_OPERATORS[type(node.op)](_eval(node.left), _eval(node.right))
        if isinstance(node, ast.UnaryOp):
            return _ALLOWED_OPERATORS[type(node.op)](_eval(node.operand))
        raise ValueError("Expression non autorisée")
    tree = ast.parse(expression, mode="eval")
    return _eval(tree.body)


## Bloc 3 — Tester chaque outil isolément

**Toujours tester une brique seule avant de l'intégrer dans une boucle plus
complexe** — c'est ce qui nous aurait fait gagner du temps lors du débogage
du notebook original. Ici, on vérifie que les deux fonctions marchent, sans
passer par Claude.


In [ ]:
# Test 1 : la recherche Wikipédia
resultat_recherche = search_wikipedia("Albert Einstein")
print("Test search_wikipedia :")
print(resultat_recherche[:200], "...\n")

# Test 2 : le calcul sécurisé
resultat_calcul = safe_calculate("120000 / 1000")
print("Test safe_calculate : 120000 / 1000 =", resultat_calcul)

# Test 3 : le calcul rejette une expression dangereuse
try:
    safe_calculate("__import__('os').system('echo test')")
    print("ATTENTION : une expression dangereuse est passée, à corriger !")
except Exception as e:
    print("Test sécurité OK — expression rejetée :", e)


## Bloc 4 — Déclarer les outils pour Claude (tool use) et le routeur

Le `TOOLS_SCHEMA` décrit à Claude quels outils existent, avec un schéma
d'entrée précis (comme une signature de fonction). C'est ce qui remplace le
parsing de texte fragile (`split()` sur `"Action: Nom[argument]"`) utilisé
dans le notebook original.

`execute_tool()` fait le lien entre ce que Claude demande et le vrai code
Python à exécuter.


In [ ]:
TOOLS_SCHEMA = [
    {
        "name": "search_wikipedia",
        "description": "Cherche une entité exacte sur Wikipédia (en français) et renvoie un résumé.",
        "input_schema": {
            "type": "object",
            "properties": {
                "query": {"type": "string", "description": "Terme exact à chercher, ex: 'Albert Einstein'"}
            },
            "required": ["query"],
        },
    },
    {
        "name": "calculer",
        "description": "Effectue un calcul mathématique simple (+, -, *, /, puissance).",
        "input_schema": {
            "type": "object",
            "properties": {
                "expression": {"type": "string", "description": "Expression mathématique, ex: '120000 / 1000'"}
            },
            "required": ["expression"],
        },
    },
    {
        "name": "finish",
        "description": "Termine la tâche et donne la réponse finale.",
        "input_schema": {
            "type": "object",
            "properties": {"answer": {"type": "string"}},
            "required": ["answer"],
        },
    },
]


def execute_tool(tool_name: str, tool_input: dict) -> str:
    if tool_name == "search_wikipedia":
        return search_wikipedia(tool_input["query"])
    if tool_name == "calculer":
        try:
            return str(safe_calculate(tool_input["expression"]))
        except Exception as e:
            return f"Erreur de calcul : {e}"
    return f"Erreur : outil '{tool_name}' inconnu."


print("TOOLS_SCHEMA et execute_tool() prêts —", len(TOOLS_SCHEMA), "outils déclarés.")


## Bloc 5 — Tester le routeur, sans encore appeler Claude

On simule un appel d'outil "à la main" pour vérifier que `execute_tool`
route bien vers la bonne fonction, avant d'ajouter la couche Claude par-dessus.


In [ ]:
print(execute_tool("calculer", {"expression": "42 * 2"}))
print(execute_tool("search_wikipedia", {"query": "Ulm"})[:150])
print(execute_tool("outil_qui_n_existe_pas", {}))


## Bloc 6 — La boucle ReAct (Thought → Action → Observation → Finish)

C'est le cœur de l'agent. À chaque tour :
1. On envoie l'historique complet à Claude
2. Claude répond, éventuellement avec un appel d'outil (`tool_use`)
3. Si c'est `finish`, on s'arrête et on renvoie la réponse
4. Sinon, on exécute réellement l'outil, et on renvoie le résultat à Claude
   sous forme de `tool_result` (pas de texte à re-parser)


In [ ]:
SYSTEM_PROMPT = """Tu résous des questions en utilisant les outils à ta disposition.
Réfléchis brièvement avant chaque appel d'outil, puis appelle l'outil approprié.
Quand tu as la réponse finale, appelle l'outil 'finish'.
"""


def run_react_tool_use(question: str, max_steps: int = 6, verbose: bool = True) -> str:
    messages = [{"role": "user", "content": question}]

    for step in range(max_steps):
        response = client.messages.create(
            model="claude-sonnet-4-6",
            max_tokens=500,
            system=SYSTEM_PROMPT,
            tools=TOOLS_SCHEMA,
            messages=messages,
        )

        for block in response.content:
            if block.type == "text" and verbose:
                print(f"--- Étape {step + 1} : Thought ---\n{block.text}\n")

        tool_use_block = next((b for b in response.content if b.type == "tool_use"), None)

        if tool_use_block is None:
            messages.append({"role": "assistant", "content": response.content})
            messages.append({"role": "user", "content": "Utilise un outil pour continuer."})
            continue

        tool_name = tool_use_block.name
        tool_input = tool_use_block.input
        if verbose:
            print(f"--- Étape {step + 1} : Action ---\n{tool_name}({tool_input})\n")

        if tool_name == "finish":
            return tool_input["answer"]

        observation = execute_tool(tool_name, tool_input)
        if verbose:
            print(f"--- Étape {step + 1} : Observation ---\n{observation}\n")

        messages.append({"role": "assistant", "content": response.content})
        messages.append({
            "role": "user",
            "content": [
                {"type": "tool_result", "tool_use_id": tool_use_block.id, "content": observation}
            ],
        })

    return "Nombre maximum d'étapes atteint sans réponse finale."


print("Fonction run_react_tool_use() définie.")


## Bloc 7 — Test de bout en bout

On pose une question qui **nécessite les deux outils dans l'ordre** :
d'abord une recherche (trouver la ville de naissance d'Einstein), puis une
seconde recherche (sa population), puis un calcul. C'est un bon test parce
qu'il force l'agent à enchaîner plusieurs tours de la boucle, pas un seul.

**Résultat attendu** : vous devez voir plusieurs blocs Thought/Action/Observation
s'afficher, se terminant par une réponse finale cohérente.


In [ ]:
question = (
    "Quelle est la population de la ville de naissance d'Albert Einstein, "
    "divisée par 1000 ?"
)

reponse = run_react_tool_use(question)
print("=" * 50)
print("Réponse finale :", reponse)


## Bloc 8 — Pour aller plus loin (exercices)

1. **Ajoutez un troisième outil** (ex. une conversion d'unités, une recherche
   météo) et observez comment l'agent l'intègre sans que la boucle change.
2. **Cassez volontairement `safe_calculate`** (retirez la sécurité) et testez
   avec une question piège pour comprendre le risque du `eval()` brut.
3. **Changez `max_steps` à 2** sur une question qui a besoin de 3 étapes —
   observez le message de repli ("Nombre maximum d'étapes atteint...").
4. **Comparez avec le repo original `ysymyth/ReAct`** : quelles erreurs
   rencontrées là-bas (parsing, `stop_sequences`) n'existent plus ici, et pourquoi ?

## Ce qu'on a appris

- Le pattern ReAct (Thought/Action/Observation) est un principe, pas un outil
  particulier — il s'implémente différemment selon les contraintes de l'API
- Le tool use structuré élimine une classe entière de bugs (parsing de texte)
- Tester chaque brique isolément (recherche, calcul, routeur) avant de les
  assembler dans la boucle complète est ce qui rend le débogage gérable
